In [ ]:
from operator import itemgetter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import math

In [59]:
from enum import Enum
class PredictionType(Enum):
    CLASSIFICATION = 1
    REGRESSION = 2

In [60]:
class SimpleKNN:
    
    def fit(self, X_train: np.matrix, y_train: np.ndarray):
        """
        Метод обучения, который фактически не учится, 
        а только запоминает обучающую выборку.
        Входные параметры:
        X_train - обучающая выборка (матрица объект-признак)
        y_train - обучающая выборка (вектор целевого признака)
        Возвращаемое значение: нет
        """
        #Сохраняем параметры в переменных класса
        self._X_train = X_train
        self._y_train = y_train
          
    def eucl_dist(self, p: np.ndarray, q: np.ndarray) -> float:
        """
        Вычисление Евклидова расстояния - https://en.wikipedia.org/wiki/Euclidean_distance 
        Входные параметры:
        p, q - вектора в n-мерном пространстве признаков
        """
        return math.sqrt(sum([(pi - qi) ** 2 for pi, qi in zip (p, q)]))
            
            
    def predict_for_single_object(self, K: int, \
                prediction_type: PredictionType, \
                X_o: np.ndarray, \
                verbose = True) -> np.ndarray:
        """
        Метод предсказания для одного объекта.
        Входные параметры:
        K - гиперпараметр, количество соседей 
        prediction_type - классификация или регрессия 
        X_o - строка матрицы объект-признак, соответствующая объекту
        verbose - флаг детального вывода
        Возвращаемое значение: предсказанное значение целевого признака
        """
        # список соседей
        neighbors_list = []
        # *** Находим ближайшие точки ***
        # Перебираем все точки обучающей выборки
        for i in range(self._X_train.shape[0]):
            # получаем текущую точку
            data_train_current_x = self._X_train.iloc[i].values
            # и значение ее y
            data_train_current_y = self._y_train[i]
            # вычисляем расстояние
            dist = self.eucl_dist(X_o, data_train_current_x)
            # сохраняем в список соседей
            temp_res = (data_train_current_y, dist, data_train_current_x)
            neighbors_list.append(temp_res)
        # *** сортируем список соседей по возрастанию расстояния *** 
        # в кортеже элементы следуют в порядке (0,1,2), сортируем по первому элементу 
        neighbors_list_sorted = sorted(neighbors_list, key=itemgetter(1))
        if verbose:
            print('Вывод К ближайших соседей:')
            for cur_y, cur_dist, temp_x in K_neighbors_list_sorted:
                print(
                    f'y={cur_y}, расстояние={cur_dist:.2f}'
                )
        # Оставим только K ближайших соседей
        K_neighbors_list_sorted = neighbors_list_sorted[:K]
        if verbose:
            print('Вывод К ближайших соседей:')
            x1_list = []
            x2_list = []
            for cur_y, cur_dist, temp_x_1_2 in K_neighbors_list_sorted:
                temp_x1, temp_x2 = temp_x_1_2
                x1_list.append(temp_x1)
                x2_list.append(temp_x2)
                print('X1={0}, X2={1}, y={2}, расстояние={3:.2f}'.format(temp_x1, temp_x2, cur_y, cur_dist))
            print()
            print('Визуализация К ближайших соседей:')
            plt.plot(self._X_train['x1'], self._X_train['x2'], 'b.', \
                     x1_list, x2_list,  'g*', \
                    [X_o[0]], [X_o[1]], 'ro')
            plt.show()   
        # Результат - классификация или регрессия
        if prediction_type == PredictionType.REGRESSION:
            # используем numpy для вычисления среднего значения
            arr = np.array([x for x,_,_ in K_neighbors_list_sorted])
            # возвращаем среднее значение
            return np.mean(arr)          
        elif prediction_type == PredictionType.CLASSIFICATION:
            k_y_list = [y for y,_,_ in K_neighbors_list_sorted]
            # группируем с количеством метки классов,
            # соответствующие K ближайшим соседям
            k_y_list_grouped_temp = np.unique(k_y_list, return_counts=True)
            k_y_list_grouped = [[key, cnt] for key, cnt in zip(k_y_list_grouped_temp[0], k_y_list_grouped_temp[1])]
            # сортируем по количеству по убыванию
            k_y_list_grouped_sorted = sorted(k_y_list_grouped, key=itemgetter(1), reverse=True)
            if verbose:
                print('Классы, соответствующие К ближайшим соседям:')
                for i in k_y_list_grouped_sorted:
                    print('класс={0}, количество элементов={1}'.format(i[0], i[1]))
            # возвращаеv метку класса из первой строки отсортированного массива
            # то есть того класса, к которому принадлежит наибольшее количество соседей
            return k_y_list_grouped_sorted[0][0]
        else:
            raise Exception('Неизвестный тип предсказания')
                   
    
    def predict(self, K: int, \
                prediction_type: PredictionType, \
                X_test: np.matrix, 
                verbose = True) -> np.ndarray:
        """
        Метод предсказания.
        Входные параметры:
        K - гиперпараметр, количество соседей 
        prediction_type - классификация или регрессия 
        X_test - тестовая выборка (матрица объект-признак)
        Возвращаемое значение: предсказанный вектор целевого признака
        """
        # Перебираем все точки тестовой выборки
        test_data_temp = []
        for i in range(X_test.shape[0]):
            # получаем текущую точку
            data_test_current_x = [x for x in X_test.iloc[i]]
            test_data_temp.append(data_test_current_x)       
        return [self.predict_for_single_object(K=K, \
                prediction_type=prediction_type, \
                X_o=i, verbose=verbose) for i in test_data_temp]
    

In [61]:
import pandas as pd

df = pd.read_csv('GPU_benchmarks_v7.csv')
df = df.drop('gpuName', axis=1)
df['category'] = df['category'].astype('category').cat.codes

In [62]:
df.isnull().sum() #пропущенные значения

G3Dmark                0
G2Dmark                0
price               1764
gpuValue            1764
TDP                 1625
powerPerformance    1625
testDate               0
category               0
dtype: int64

In [63]:
df = df.fillna(df.mean(numeric_only=True))

In [64]:
print(df.isnull().sum())

G3Dmark             0
G2Dmark             0
price               0
gpuValue            0
TDP                 0
powerPerformance    0
testDate            0
category            0
dtype: int64


Формируем признаки и цель. Например, будет предсказывать цену


In [65]:
X = df.drop('price', axis=1)
y = df['price']

### 3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [67]:
#Обучение собственной модели
knn = SimpleKNN()

knn.fit(X_train, y_train.values)

### 4. Обучите модель ближайших соседей для произвольно заданного гиперпараметра K. Оцените качество модели с помощью подходящих для задачи метрик.

In [68]:
y_pred = knn.predict(
    K=5,
    prediction_type=PredictionType.REGRESSION,
    X_test=X_test,
    verbose=False
)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


#Оценка качества
print("MAE =", mean_absolute_error(y_test, y_pred))
print("MSE =", mean_squared_error(y_test, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 =", r2_score(y_test, y_pred))

MAE = 151.50289333000768
MSE = 275459.05702101026
RMSE = 524.841935272907
R2 = 0.004513514996538293


### 5. Подбор оптимального K через GridSearchCV и RandomizedSearchCV.

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [73]:
#GRIDSEARCHCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    'n_neighbors': range(1, 31)
}

cv1 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param_grid,
    cv=cv1,
    scoring='neg_mean_squared_error'
)

grid.fit(X_train_scaled, y_train)

print("Лучший K:", grid.best_params_)
print("Лучший score:", grid.best_score_)

Лучший K: {'n_neighbors': 2}
Лучший score: -33441.4629650831


In [75]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_grid = grid.best_estimator_

y_pred_grid = best_grid.predict(X_test_scaled)

print("MAE =", mean_absolute_error(y_test, y_pred_grid))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_grid)))
print("R2 =", r2_score(y_test, y_pred_grid))

MAE = 83.62324425547173
RMSE = 369.17547494931677
R2 = 0.507457175616563


In [ ]:
#Для Randomized
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=KNeighborsRegressor(),
    param_distributions={
        'n_neighbors': range(1, 31)
    },
    n_iter=20,
    cv=cv1,
    scoring='neg_mean_squared_error',
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print("Лучший K:", random_search.best_params_)

Лучший K: {'n_neighbors': 2}


In [77]:
best_random = random_search.best_estimator_

y_pred_random = best_random.predict(X_test_scaled)

print("MAE =", mean_absolute_error(y_test, y_pred_random))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_random)))
print("R2 =", r2_score(y_test, y_pred_random))

MAE = 83.62324425547173
RMSE = 369.17547494931677
R2 = 0.507457175616563


In [79]:
#Вторая стратегия кросс-валидации

from sklearn.model_selection import RepeatedKFold

cv2 = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

grid2 = GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    cv=cv2,
    scoring='neg_mean_squared_error'
)

grid2.fit(X_train_scaled, y_train)

print(grid2.best_params_)

{'n_neighbors': 2}


In [80]:
best_grid2 = grid2.best_estimator_

y_pred_grid2 = best_grid2.predict(X_test_scaled)

print("MAE =", mean_absolute_error(y_test, y_pred_grid2))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_grid2)))
print("R2 =", r2_score(y_test, y_pred_grid2))

MAE = 83.62324425547173
RMSE = 369.17547494931677
R2 = 0.507457175616563


In [ ]:
print("\nСРАВНЕНИЕ МОДЕЛЕЙ\n")

print("БАЗОВАЯ МОДЕЛЬ (K=5, self-made KNN)")
print("MAE =", mean_absolute_error(y_test, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 =", r2_score(y_test, y_pred))

print("\n GRIDSEARCHCV (оптимальный K)")
print("Best K =", grid.best_params_)

print("MAE =", mean_absolute_error(y_test, y_pred_grid))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_grid)))
print("R2 =", r2_score(y_test, y_pred_grid))

print("\n RANDOMIZEDSEARCHCV")
print("Best K =", random_search.best_params_)

print("MAE =", mean_absolute_error(y_test, y_pred_random))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_random)))
print("R2 =", r2_score(y_test, y_pred_random))

print("\n GRIDSEARCH + RepeatedKFold")
print("Best K =", grid2.best_params_)

print("MAE =", mean_absolute_error(y_test, y_pred_grid2))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred_grid2)))
print("R2 =", r2_score(y_test, y_pred_grid2))